# Dr Amato - Orbital Mech Tutorials, Tutorial 1
Lecture 5 - Coordinate Systems, Orbital Elements and Orbit Propagation

## Question 1

### Part a

In [ ]:
# Variable loading
import numpy as np

# Initialising the variable
r = np.array([
    1.073,
    1.514,
    0.541
])

# WIP

# Calculating the RA
RA = np.atan(r[1] / r[0]) * 180/np.pi

# Want to check the quadrant state
#   Quadrant 1, add 0 degrees
if r[0] > 0 and r[1] > 0:
    RA += 0

#   Quadrant 2, add 90 degrees
elif r[0] < 0 and r[1] > 0:
    RA += 90

#   Quadrant 3, add 180 degrees
elif r[0] < 0 and r[1] < 0:
    RA += 180

#   Quadrant 4, add 270 degrees
elif r[0] > 0 and r[1] < 0:
    print("False")



RA = np.atan(r[1] / r[0]) * 180/np.pi
print(RA)

# Calculating the DE
xylength = np.sqrt(r[0]**2 + r[1]**2)
DE = np.atan(r[2] / xylength) * 180/np.pi
print(DE)

True
54.674089143953495
16.253427217902644


### Part b

Solution consists of a rotation matrix in the x axis

### Part c

In [8]:
import numpy as np

# Initialising the variable
r = np.array([
    1.073,
    1.514,
    0.541
])

# Obliquity value
varepsilon = 23.4
#   Converted to radians
varepsilon *= np.pi/180

# Rotationg from ECI to ECIE
ECI_to_ECIE = np.array([
    [1, 0, 0,],
    [0, np.cos(varepsilon), np.sin(varepsilon)],
    [0, -np.sin(varepsilon), np.cos(varepsilon)],
])

# Position vector of asteroid in ECIE frame
r_ECIE = np.linalg.matmul(ECI_to_ECIE, r)
print("Rotated value = ", r_ECIE)

Rotated value =  [ 1.073       1.60433751 -0.10477665]


## Question 2

Solution is a written proof, will be written down later

## Question 3

Solution is a written proof, will be written down later

## Question 4

In [27]:
import numpy as np
import time
import requests

# Change option from a, b or c for your respective part
opt = "a"

# Solving part a, b and c
if opt == "a":
    y, m, d = 2008, 1, 1
    UT = 12

    # Positive as it is due eastward
    Lambda1, Lambda2 = 18, 3

elif opt == "b":
    y, m, d = 2005, 7, 4
    UT = 20

    # Negative due to being westward
    Lambda1, Lambda2 = -118, -15

elif opt == "c":
    # This option gets current time and location of where you are based when you executed this code block
    print("Current date:")
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "\n")

    # Gets the year, month and day of current time
    y, m, d = int(time.strftime("%Y")), int(time.strftime("%m")), int(time.strftime("%d"))
    # Gets current time and converts into decimal value
    UT = int(time.strftime("%H")) + int(time.strftime("%M")) / 60 + int(time.strftime("%S")) / 3600

    # Uses requests to get the longitude of place
    # Will require an internet connection so if there is no internet, then will default to Greenwhich (longitude 0)
    try:

        # Collects information using ipinfo.io
        response = requests.get("https://ipinfo.io/json")
        data = response.json()

        # Since we have the lontitude as a decimal, we do not need Lamdba2
        lattitude, Lambda1 = data["loc"].split(",")
        Lambda1 = float(Lambda1)
        Lambda2 = 0

        # No interest in lattitude so we disregard lattitude
        latittude = None

    except:
        print("Error, cannot connect to ipinfo.io")
        print("Will default to the same Greenwhich position (0 longitude)")
        Lambda1 = 0
        Lambda2 = 0

Lambda = Lambda1 + Lambda2/60

## J0
JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24

## T0
T0 = (JD - 2451545) / 36525
print(T0)

## theta G0
theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
print(theta_G0)

## theta G
theta_G =  theta_G0 + 360.98564724 * UT/24
while theta_G > 360:
    theta_G -= 360

print(theta_G)

## Local Sideral Time
LocalSiderealTime = theta_G + Lambda
print(LocalSiderealTime)


0.07997262149212868
2979.536576715145
280.02940033514506
298.07940033514507


## Question 5

### Part a

In [3]:
import time
import requests
import numpy as np

def gettheta_G():
    print("Current date:")
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "\n")

    y, m, d = int(time.strftime("%Y")), int(time.strftime("%m")), int(time.strftime("%d"))
    UT = int(time.strftime("%H")) + int(time.strftime("%M")) / 60 + int(time.strftime("%S")) / 3600

    try:
        response = requests.get("https://ipinfo.io/json")
        data = response.json()
        lattitude, Lambda1 = data["loc"].split(",")
        Lambda1 = float(Lambda1)
        Lambda2 = 0

        latittude = None

    except:
        print("Error, cannot connect to ipinfo.io")
        print("Will default to the same Greenwhich position (0 longitude)")
        Lambda1 = 0
        Lambda2 = 0

    Lambda = Lambda1 + Lambda2/60

    JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24
    T0 = (JD - 2451545) / 36525
    theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
    theta_G =  theta_G0 + 360.98564724 * UT/24
    while theta_G > 360:
        theta_G -= 360

    print("Theta_G = ", theta_G, "degrees")
    return theta_G * np.pi/180

# Get the theta_G of the current time and place
theta_G = gettheta_G()

# Apply matrix transformation from ECEF to ECI
ECI_to_ECEF = np.array([
    [np.cos(theta_G), np.sin(theta_G), 0],
    [-np.sin(theta_G), np.cos(theta_G), 0],
    [0, 0, 1]
])
ECEF_to_ECI = np.linalg.inv(ECI_to_ECEF)
print(ECEF_to_ECI)

# Getting current r_ECEF
# Want to get the current lattitude and longitude angles
try:
    response = requests.get("https://ipinfo.io/json")
    data = response.json()
    lattitude, longitude = data["loc"].split(",")
    lattitude, longitude = float(lattitude), float(longitude)

except:
    print("Error, cannot connect to ipinfo.io")
    print("Will default to the same Greenwhich position (0 longitude, 51.58 lattitude)")
    longitude = 0
    lattitude = 51.58

# Convert lattitude and longitude from degrees to radians
lattitude *= np.pi/180
longitude *= np.pi/180

# Make the assumption of r as 6370, and then get the r_ECEF
r = 6370
r_ECEF = np.array([
    r * np.sin(lattitude) * np.cos(longitude),
    r * np.sin(lattitude) * np.sin(longitude),
    r * np.cos(lattitude)
])
print("r_ECEF coordinates:")
print(r_ECEF)

# Apply matrix transformation to get your ECI frame of your current position
r_ECI = np.linalg.matmul(ECEF_to_ECI, r_ECEF)
print("r_ECI: \n", r_ECI)

Current date:
2026-09-02 19:56:18 

Theta_G =  279.9001993062702 degrees
[[ 0.17193253  0.98510873  0.        ]
 [-0.98510873  0.17193253 -0.        ]
 [ 0.          0.          1.        ]]
r_ECEF coordinates:
[4985.79018734  -10.93823729 3964.67862037]
r_ECI: 
 [  846.44415313 -4913.42606872  3964.67862037]


### Part b

For this case, you should apply the rotation velocity on $r_{ECI}$.

$$
\underline v_{ECI} = \omega \times r_{ECI}
$$

Assume the angular velocity is $7.292115 \times 10^{-5} rad/s$, the proof for this value can be done as an exercise for the viewer

In [10]:
import time
import requests
import numpy as np

def gettheta_G():
    print("Current date:")
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "\n")

    y, m, d = int(time.strftime("%Y")), int(time.strftime("%m")), int(time.strftime("%d"))
    UT = int(time.strftime("%H")) + int(time.strftime("%M")) / 60 + int(time.strftime("%S")) / 3600

    try:
        response = requests.get("https://ipinfo.io/json")
        data = response.json()
        lattitude, Lambda1 = data["loc"].split(",")
        Lambda1 = float(Lambda1)
        Lambda2 = 0

        latittude = None

    except:
        print("Error, cannot connect to ipinfo.io")
        print("Will default to the same Greenwhich position (0 longitude)")
        Lambda1 = 0
        Lambda2 = 0

    Lambda = Lambda1 + Lambda2/60

    JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24
    T0 = (JD - 2451545) / 36525
    theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
    theta_G =  theta_G0 + 360.98564724 * UT/24
    while theta_G > 360:
        theta_G -= 360

    print("Theta_G = ", theta_G, "degrees")
    return theta_G * np.pi/180

# Get the theta_G of the current time and place
theta_G = gettheta_G()

# Apply matrix transformation from ECEF to ECI
ECI_to_ECEF = np.array([
    [np.cos(theta_G), np.sin(theta_G), 0],
    [-np.sin(theta_G), np.cos(theta_G), 0],
    [0, 0, 1]
])
ECEF_to_ECI = np.linalg.inv(ECI_to_ECEF)

# Getting current r_ECEF
# Want to get the current lattitude and longitude angles
try:
    response = requests.get("https://ipinfo.io/json")
    data = response.json()
    lattitude, longitude = data["loc"].split(",")
    lattitude, longitude = float(lattitude), float(longitude)

except:
    print("Error, cannot connect to ipinfo.io")
    print("Will default to the same Greenwhich position (0 longitude, 51.58 lattitude)")
    longitude = 0
    lattitude = 51.58

# Convert lattitude and longitude from degrees to radians
lattitude *= np.pi/180
longitude *= np.pi/180

# Make the assumption of r as 6370, and then get the r_ECEF
r = 6370
r_ECEF = np.array([
    r * np.sin(lattitude) * np.cos(longitude),
    r * np.sin(lattitude) * np.sin(longitude),
    r * np.cos(lattitude)
])

# Apply matrix transformation to get your ECI frame of your current position
r_ECI = np.linalg.matmul(ECEF_to_ECI, r_ECEF)

# Angular velocity
w = 7.292115e-5

# Final velocity 
v_ECI = np.array([
    r_ECI[0],
    r_ECI[1],
    r_ECI[2] * w
])

print(v_ECI)

Current date:
2026-09-02 20:49:57 

Theta_G =  293.386143720114 degrees
[ 1.96894975e+03 -4.58055241e+03  2.89108924e-01]


## Question 6

Solution, follow a multitude of steps for you to get all orbital elements
1. Get magnitude of $|\underline{r}|$ and $|\underline{v}|$

2. Get $\underline{h}$ and magnitude of $|\underline h|$
$$ \underline h = \underline r \times \underline v $$

3. Find SMA
- from vis viva equation
$$\varepsilon = \frac{v^2}{2} - \frac{\mu}{r}$$
- Making the subsitution of $\varepsilon = \frac{-\mu}{2a}$
$$\frac{v^2}{2} = \frac{\mu}{r} - \frac{\mu}{a}$$
- Finally rearranging to get SMA
$$a = \frac{\mu}{\frac{2\mu}{r} - v^2}$$

4. Find ECC and magnitude $|\underline e |$
$$\underline e = \frac{1}{\mu} [\underline v \times \underline h - \mu \frac{\underline r}{r}]$$

5. Find the inclination angle
$$i = acos\left(\frac{h_z}{h}\right)$$

6. Find the nodal vector and the magnitude
$$\underline N = i_z \times \underline h$$

7. Find the RAAN
- **if N_y > 0** :
    $$\Omega = \arccos(\frac{N_x}{N})$$
- else
    $$\Omega = 360 - \arccos(\frac{N_x}{N})$$


8. Find the AOP
- **if e_z > 0**:
    $$\omega = \arccos(\frac{\underline N \cdot \underline e}{|\underline N||\underline e|})$$
- else:
    $$\omega = 360 - \arccos(\frac{\underline N \cdot \underline e}{|\underline N||\underline e|})$$


9. Find the True Anomaly

- **if $\underline r \cdot \underline v > 0$** :
    $$\theta = \arccos(\frac{\underline r \cdot \underline e}{|\underline r||\underline e|})$$
- else:
    $$\theta = 360 - \arccos(\frac{\underline r \cdot \underline e}{|\underline r||\underline e|})$$



In [ ]:
import numpy as np
from math import cos, acos, pi

# constants
mu = 1

# User parameters
opt = 1
if opt == 1:
    # q1 
    r = np.array([0.9412, -5.9851, 8.3585])
    v = np.array([0.4365, 0.127, 0.5661])
elif opt == 2:
    r = np.array([-6.0955, -11.8452, 11.0082])
    v = np.array([-0.0153, -0.7556, -0.044])
elif opt == 3:
    r = np.array([-1.7371, 0.0256, 0.1635])
    v = np.array([-0.0153, -0.7556, -0.0440])
elif opt == 4:
    r = np.array([-6200, -3560, 3100])
    v = np.array([-3.4, 6.3, 2.5])
elif opt == 5:
    r = np.array([
        -10527 + 12/99 * -1559,
        3039 + 12/99 * 450,
        5264 + 12/99 * 779
    ])
    v = np.array([
        -4.959,
        -1.432,
        -2.480
    ])

# Magnitude of r and v
R = np.linalg.norm(r)
V = np.linalg.norm(v)
print("r mag = " , R)
print("v mag = " , V)

# angular momentum
h = np.cross(r, v)
H = np.linalg.norm(h)
print("h = " , h)
print("h mag = " , H)


# semi major axis
a = mu / (2 * mu / R - (V)**2)
print("a = " , a)

# Eccentricity
e = 1/mu * (np.cross(v, h) - mu * r / R)
emag = np.linalg.norm(e)
print(np.cross(v, h))
print("e = ", e)
print("e mag = ", emag)

# inclination
i = acos(h[2] / H) * 180 / pi
print("i = ",i)

# Nodal vector
N = np.cross([0, 0, 1], h)
Nmag = np.linalg.norm(N)
print("N = ", N)
print("N magnitude = " , Nmag)

# RAAN
if N[1] > 0:
    print("bug")
    Omega = acos(N[0] / Nmag) * 180 / pi
else:
    Omega = 360 - acos(N[0] / Nmag) * 180 / pi
print("Omega = " , Omega)

# AOP
if e[2] > 0:
    omega = acos(np.dot(N, e) / (Nmag * emag)) * 180 / pi
else:
    omega = 360 - acos(np.dot(N, e) / (Nmag * emag)) * 180 / pi
print("omega = ", omega)

# theta
if np.dot(r, v) > 0:
    theta = acos(np.dot(r, e) / (R * emag)) * 180 / pi
else:
    theta = 360 -  acos(np.dot(r, e) / (R * emag)) * 180 / pi
print("theta =", theta)

print("Final results:")
print("a = ", a)
print(e)
print(i)
print(Omega)
print(omega)
print(theta)

r mag =  10.323361937857259
v mag =  0.7260375059182549
h =  [-4.44969461  3.11567193  2.73202855]
h mag =  6.080392560983878
a =  -2.9994439925766825
[-1.41681425 -3.71150258  1.92510201]
e =  [-1.5079861  -3.1317399   1.11543361]
e mag =  3.6504806554836446
i =  63.30007749090055
N =  [-3.11567193 -4.44969461  0.        ]
N magnitude =  5.4320524387797455
Omega =  235.0003264146696
omega =  20.00046359968696
theta = 44.999294933139446
Final results:
a =  -2.9994439925766825
[-1.5079861  -3.1317399   1.11543361]
63.30007749090055
235.0003264146696
20.00046359968696
44.999294933139446


## Question 7
Orbital elements into radial and velocity vectors

Solution, follow the multitude of steps for you to find the final answer.
The more or less roundabout is to find the value of $|\underline v_p|$ and $|\underline r_p|$ through eccentricity equation.

1. Find the true anomaly from M.
$$E = M + e \sin E$$
$$\tan \frac{\theta}{2} = \tan \frac{E}{2} \sqrt{\frac{1 + e}{1 - e}}$$

2. Find the angular momentum
$$\frac{h^2}{2} = a (1 - e^2)$$

3. Find the radius and velocity vectors in the **perifocal** frame
$$r = \frac{h^2}{\mu (1 + e \cos \theta)} (\cos \theta i_e + \sin \theta i_p)$$
$$v = \frac{\mu}{h} (-e \sin \theta i_e + (e + \cos \theta) i_e)$$

4. Use the transformation matrix from perifocal frame to ECI frame, check the data sheet

Tutorial solution

In [ ]:
import numpy as np
from math import cos, radians, sin, atan, degrees, tan, sqrt

a = 4.58
e = 0.8
i = 87
Omega = 93
omega = 202
M = 90

# Normalised question
mu = 1

# Find true anomaly
E = 1
for j in range(1, 200):
    E = radians(M) + e * sin(E)
print("E = ", E, "rads (or ", degrees(E), "deg)")

theta = 2 * atan( tan(E/2) * sqrt((1 + e) / (1 - e)))
print("theta = ", theta, "rads (or ", degrees(theta), "deg)")

# Find the angular momentum
h = sqrt(mu * a * ( 1 - e**2))
print("h = ", h)

# Find r and v in perifocal frame
r = np.array(
    [h**2 / mu * 1 / (1 + e * cos(theta)) * cos(theta), # i_e vector
    h**2 / mu * 1 / (1 + e * cos(theta)) * sin(theta), # i_p vector
    0]                                                 # i_h vector
)
print("r = ", r)

v = np.array(
    [mu / h * ( -np.sin(theta)) , # i_e vector
    mu / h * (e + np.cos(theta)), # i_p vector
    0]                                                 # i_h vector
)
print("v = ", v)

# Apply perifocal to ECI transformation
cO = np.cos(np.radians(Omega))
sO = np.sin(np.radians(Omega))
co = np.cos(np.radians(omega))
so = np.sin(np.radians(omega))
ci = np.cos(np.radians(i))
si = np.sin(np.radians(i))

perifocal_to_inertial_frame = np.array([
    [
        cO*co - sO*so*ci,
        -cO*so - sO*co*ci,
        sO*si
    ],
    [
        sO*co + cO*so*ci,
        -sO*so + cO*co*ci,
        -cO*si
    ],
    [
        so*si,
        co*si,
        ci
    ]
])

print("Transformation matrix:\n ", perifocal_to_inertial_frame)

r = perifocal_to_inertial_frame @ r
v = perifocal_to_inertial_frame @ v
print("\n\nFinal vectors:\n")
print("r: ", r, "\n")
print("v: ", v)

E =  2.2119306096084457 rads (or  126.73428850636327 deg)
theta =  2.81033528305589 rads (or  161.0203507358061 deg)
h =  1.2840560735419615
r =  [-6.40332026  2.20229626  0.        ]
v =  [-0.25328512 -0.11341728  0.        ]
Transformation matrix:
  [[ 0.06810358  0.02885316  0.99726095]
 [-0.92488711  0.37663281  0.05226423]
 [-0.37409321 -0.92591318  0.05233596]]


Final vectors:

r:  [-0.37254583  6.75180543  0.35630348] 

v:  [-0.02052207  0.19154347  0.1997668 ]


Calculator (used for mock exam)

In [ ]:
import numpy as np
from math import radians

a = 26610.21
e = 0.747
theta = 24.31

# Non-normalised question
mu = 398600

# Find the eccentric anomaly
E = 2 * np.arctan(np.sqrt((1 - e) / (1 + e)) * np.tan(radians(theta)/2))
print("E = ", E)

# Find the mean anomaly
M = E - e * np.sin(E)
print("M = ", M)

# Find the time taken

## Periapsis
periapsis_t = M * np.sqrt(a**3 / mu)
## Apoapsis
apoapsis_t = (np.pi - M) * np.sqrt(a**3 / mu)

print(f"Time taken from periapsis: {periapsis_t}s or {periapsis_t / 3600} hrs")
print(f"Time taken from apoapsis: {apoapsis_t}s or {apoapsis_t / 3600} hrs")

E =  0.1635651860122654
M =  0.04192606985663318
Time taken from periapsis: 288.2623741991408s or 0.08007288172198357 hrs
Time taken from apoapsis: 21311.734004834132s or 5.9199261124539255 hrs
